# Week 10: Rotational Dynamics (II) — Angular Momentum — PHASE 4: Rotation — Same Laws, New Geometry

*📚 Physics I (PHY101) · ⏱️ 3 Hours · 👨‍🏫 Dr. Arif Solmaz*

## 🎯 Learning Objectives

By the end of this week, you will be able to:

1. **Define** angular momentum for a rigid body and a particle
2. **Apply** conservation of angular momentum to systems with no external torque
3. **Explain** how changing moment of inertia affects angular velocity (ice skater effect)
4. **Analyze** gyroscopic precession qualitatively and quantitatively
5. **Relate** torque to the time rate of change of angular momentum ($\tau = dL/dt$)
6. **Solve** collision problems involving angular momentum conservation

## 🎯 Core Mastery Connection

Angular momentum $L = I\omega$ is conserved when no external torque acts. This is the rotational twin of linear momentum conservation. You can now predict spinning behavior: diagram the system, identify whether external torques exist, write the conservation equation, and predict the final angular velocity. The same Diagram → Principle → Equation → Prediction → Verify workflow applies.

---
## 🧭 Three-Hour Interactive Studio Plan

**Audience:** Mechatronics Engineering and Computer Engineering students  
**Weekly focus:** Week 10: Rotational Dynamics (II) — Angular Momentum — PHASE 4: Rotation — Same Laws, New Geometry

- **Mechatronics lens:** gyroscopes, reaction wheels, and angular-momentum control.
- **Computer Engineering lens:** IMU processing and orientation systems.

| Time | Learning cycle |
|---|---|
| 00:00–00:10 | Launch question, prior-knowledge retrieval, outcomes |
| 00:10–00:50 | Concept cycle 1: explain → predict → test |
| 00:50–01:00 | Checkpoint 1, student questions, peer explanation |
| 01:00–01:10 | Break |
| 01:10–01:50 | Concept cycle 2: worked example → variation → discussion |
| 01:50–02:00 | Checkpoint 2 and misconception repair |
| 02:00–02:10 | Break |
| 02:10–02:40 | Core in-class practice with instructor circulation |
| 02:40–02:50 | Checkpoint 3: exam bridge and professional transfer |
| 02:50–03:00 | Open questions, summary, and exit ticket |

The official start and finish times are followed as published in the timetable. Ask questions at any point; the scheduled checkpoints guarantee additional question time. Checkpoints are private self-checks in this runtime—no identity, upload, homework, or instructor dashboard.


In [ ]:
# Run once. This pulse stays only in the current Colab runtime.
_studio_pulses = {}

def studio_pulse(number, response, minimum_words=8):
    words = str(response).strip().split()
    ready = len(words) >= minimum_words
    _studio_pulses[int(number)] = ready
    if ready:
        print(f"✅ Checkpoint {number}: explanation recorded locally ({len(words)} words).")
    else:
        print(f"🟡 Checkpoint {number}: explain your reasoning in at least {minimum_words} words, then retry.")
    print("Nothing is transmitted or stored for grading.")
    return ready

print("✅ Local studio checkpoints ready")

---
## 1. Angular Momentum

### 1.1 Definition

**For a rigid body** rotating about a fixed axis:
$$L = I\omega$$

**For a particle** at position $\vec{r}$ with momentum $\vec{p}$:
$$\vec{L} = \vec{r} \times \vec{p} = \vec{r} \times m\vec{v}$$

| Quantity | SI Unit |
|:---|:---|
| Angular momentum $L$ | kg m$^2$/s |
| Moment of inertia $I$ | kg m$^2$ |
| Angular velocity $\omega$ | rad/s |

### 1.2 Newton's Second Law for Rotation

$$\vec{\tau}_{\text{net}} = \frac{d\vec{L}}{dt}$$

This is the **rotational equivalent** of $\vec{F} = d\vec{p}/dt$.

For a rigid body with constant $I$:
$$\tau = I\alpha$$

### 1.3 Conservation of Angular Momentum

If the net external torque on a system is zero:

$$\vec{L}_i = \vec{L}_f \quad \Rightarrow \quad I_i \omega_i = I_f \omega_f$$

> **The ice skater analogy:** When a spinning ice skater pulls their arms in, $I$ decreases. Since $L = I\omega$ must stay constant (no external torque), $\omega$ must increase. This is why they spin faster!

### 1.4 Important Comparison

| Linear | Rotational |
|:---|:---|
| $p = mv$ | $L = I\omega$ |
| $F = dp/dt$ | $\tau = dL/dt$ |
| If $F_{\text{net}} = 0$: $p$ conserved | If $\tau_{\text{net}} = 0$: $L$ conserved |
| $F\Delta t = \Delta p$ (impulse) | $\tau \Delta t = \Delta L$ (angular impulse) |

In [ ]:
# ============================================================
# Setup cell — run this first!
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display
import ipywidgets as widgets
from ipywidgets import interact, interactive, FloatSlider, IntSlider, Dropdown
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams.update({'font.size': 12, 'figure.figsize': (9, 6)})
print("All imports ready.")

---
## 🎮 Interactive Demo 1: Ice Skater Spinning — Conservation of Angular Momentum

Watch how an ice skater's angular velocity changes as they pull their arms in. The total angular momentum $L = I\omega$ stays constant throughout!

In [ ]:
# ============================================================
# DEMO 1: Ice Skater Spinning Animation
# ============================================================

def ice_skater_animation(omega_initial=2.0):
    """
    Animate an ice skater pulling arms in/out.
    Phase 1: Arms out (large I, slow spin)
    Phase 2: Arms pulling in (I decreasing, omega increasing)
    Phase 3: Arms in (small I, fast spin)
    Phase 4: Arms extending out again
    """
    # Model: body as cylinder + 2 arms as point masses
    m_body = 50.0   # kg
    r_body = 0.15   # m (body radius)
    m_arm = 4.0     # kg (each arm)
    r_arm_out = 0.7 # m (arm extended distance from axis)
    r_arm_in = 0.15 # m (arm close to body)

    I_body = 0.5 * m_body * r_body**2
    I_arms_out = 2 * m_arm * r_arm_out**2
    I_arms_in = 2 * m_arm * r_arm_in**2
    I_out = I_body + I_arms_out
    I_in = I_body + I_arms_in

    L = I_out * omega_initial  # conserved angular momentum
    omega_fast = L / I_in

    duration = 12.0
    fps = 30
    n_frames = int(duration * fps)
    times = np.linspace(0, duration, n_frames)

    # Arm radius schedule: out -> in -> out
    r_arm_schedule = np.zeros(n_frames)
    for i, t in enumerate(times):
        if t < 3:
            r_arm_schedule[i] = r_arm_out
        elif t < 5:
            frac = (t - 3) / 2.0
            r_arm_schedule[i] = r_arm_out + frac * (r_arm_in - r_arm_out)
        elif t < 8:
            r_arm_schedule[i] = r_arm_in
        elif t < 10:
            frac = (t - 8) / 2.0
            r_arm_schedule[i] = r_arm_in + frac * (r_arm_out - r_arm_in)
        else:
            r_arm_schedule[i] = r_arm_out

    # Compute I and omega for each frame
    I_schedule = I_body + 2 * m_arm * r_arm_schedule**2
    omega_schedule = L / I_schedule

    # Integrate theta
    dt = duration / n_frames
    theta_schedule = np.cumsum(omega_schedule) * dt

    fig, axes = plt.subplots(1, 3, figsize=(15, 5.5),
                              gridspec_kw={'width_ratios': [1, 1, 1.3]})
    fig.suptitle('Ice Skater — Conservation of Angular Momentum', fontsize=14, fontweight='bold')

    # --- Left panel: top view of skater ---
    ax_skater = axes[0]
    ax_skater.set_xlim(-1.0, 1.0)
    ax_skater.set_ylim(-1.0, 1.0)
    ax_skater.set_aspect('equal')
    ax_skater.set_title('Top View')
    ax_skater.set_xticks([])
    ax_skater.set_yticks([])

    body_circle = plt.Circle((0, 0), r_body, color='mediumpurple', alpha=0.7)
    ax_skater.add_patch(body_circle)
    arm_left, = ax_skater.plot([], [], 'o-', color='coral', lw=4, markersize=10)
    arm_right, = ax_skater.plot([], [], 'o-', color='coral', lw=4, markersize=10)
    skater_text = ax_skater.text(0, -0.9, '', ha='center', fontsize=10)

    # --- Middle panel: angular momentum bar ---
    ax_L = axes[1]
    ax_L.set_xlim(0, 3)
    ax_L.set_ylim(0, max(omega_schedule)*1.2)
    ax_L.set_title('Angular Momentum Decomposition')
    ax_L.set_xticks([0.5, 1.5, 2.5])
    ax_L.set_xticklabels(['$I$', '$\\omega$', '$L = I\\omega$'], fontsize=11)

    bar_I = ax_L.bar(0.5, 0, 0.6, color='steelblue', edgecolor='black', label='$I$')
    bar_omega = ax_L.bar(1.5, 0, 0.6, color='coral', edgecolor='black', label='$\\omega$')
    bar_L = ax_L.bar(2.5, 0, 0.6, color='green', edgecolor='black', label='$L$')

    # Normalize for display
    I_max = max(I_schedule)
    omega_max_val = max(omega_schedule)
    scale = omega_max_val  # use omega_max as the y-scale

    val_text_I = ax_L.text(0.5, 0, '', ha='center', va='bottom', fontsize=9)
    val_text_omega = ax_L.text(1.5, 0, '', ha='center', va='bottom', fontsize=9)
    val_text_L = ax_L.text(2.5, 0, '', ha='center', va='bottom', fontsize=9)

    # --- Right panel: time plots ---
    ax_plot = axes[2]
    ax_plot.set_xlim(0, duration)
    ax_plot.set_xlabel('Time (s)')
    ax_plot.set_title('Time Evolution')

    ax_plot2 = ax_plot.twinx()
    line_omega, = ax_plot.plot([], [], 'r-', lw=2, label=r'$\omega$ (rad/s)')
    line_I, = ax_plot2.plot([], [], 'b--', lw=2, label=r'$I$ (kg m$^2$)')
    ax_plot.set_ylabel(r'$\omega$ (rad/s)', color='red')
    ax_plot2.set_ylabel(r'$I$ (kg m$^2$)', color='blue')
    ax_plot.set_ylim(0, omega_max_val * 1.15)
    ax_plot2.set_ylim(0, I_max * 1.15)

    # L constant line
    ax_plot.axhline(y=L, color='green', ls=':', alpha=0.0)  # invisible, just for legend
    lines = [line_omega, line_I]
    labels = [l.get_label() for l in lines]
    labels.append(f'$L$ = {L:.2f} kg m$^2$/s (const)')
    ax_plot.legend(lines + [plt.Line2D([0],[0], color='green', ls=':')],
                   labels, loc='upper right', fontsize=8)

    # Phase labels
    for (t_start, t_end, label) in [(0, 3, 'Arms out'), (3, 5, 'Pulling in'),
                                     (5, 8, 'Arms in'), (8, 10, 'Extending'), (10, 12, 'Arms out')]:
        ax_plot.axvspan(t_start, t_end, alpha=0.05, color='gray')
        ax_plot.text((t_start+t_end)/2, omega_max_val*1.08, label,
                     ha='center', fontsize=7, style='italic')

    dot_omega, = ax_plot.plot([], [], 'ro', markersize=5)
    dot_I, = ax_plot2.plot([], [], 'bs', markersize=5)

    plt.tight_layout()

    def update(frame):
        t = times[frame]
        theta = theta_schedule[frame]
        r_arm = r_arm_schedule[frame]
        omega = omega_schedule[frame]
        I_curr = I_schedule[frame]

        # Skater arms (rotating reference)
        arm_left.set_data([0, r_arm * np.cos(theta)],
                          [0, r_arm * np.sin(theta)])
        arm_right.set_data([0, -r_arm * np.cos(theta)],
                           [0, -r_arm * np.sin(theta)])
        skater_text.set_text(f'$\\omega$ = {omega:.1f} rad/s\nr$_{{arm}}$ = {r_arm:.2f} m')

        # Bar chart
        I_bar_height = (I_curr / I_max) * scale
        bar_I[0].set_height(I_bar_height)
        bar_omega[0].set_height(omega)
        L_bar_height = (L / (I_max * omega_max_val / scale)) * scale * 0.5
        bar_L[0].set_height(L_bar_height)

        val_text_I.set_position((0.5, I_bar_height))
        val_text_I.set_text(f'{I_curr:.3f}')
        val_text_omega.set_position((1.5, omega))
        val_text_omega.set_text(f'{omega:.1f}')
        val_text_L.set_position((2.5, L_bar_height))
        val_text_L.set_text(f'{L:.2f}')

        # Time plots
        line_omega.set_data(times[:frame+1], omega_schedule[:frame+1])
        line_I.set_data(times[:frame+1], I_schedule[:frame+1])
        dot_omega.set_data([t], [omega])
        dot_I.set_data([t], [I_curr])

        return []

    anim = FuncAnimation(fig, update, frames=n_frames, interval=1000/fps, blit=True)
    plt.close(fig)
    return HTML(anim.to_jshtml())

ice_skater_animation(omega_initial=2.0)

**Key takeaway:** As the arms pull in, $I$ drops and $\omega$ increases so that $L = I\omega$ remains constant. The ratio $\omega_f / \omega_i = I_i / I_f$ can be very large!

---
## 2. Gyroscopic Precession

### 2.1 The Phenomenon

A spinning top or gyroscope does not simply fall over. Instead, its axis sweeps out a cone — this is **precession**.

### 2.2 Why It Precesses

Gravity exerts a torque:
$$\vec{\tau} = \vec{r} \times m\vec{g}$$

This torque is **perpendicular** to $\vec{L}$, so it changes the **direction** of $\vec{L}$, not its magnitude. The result: the angular momentum vector (and the spin axis) sweeps around in a circle.

### 2.3 Precession Rate

The precession angular velocity:

$$\Omega_{\text{prec}} = \frac{\tau}{L\sin\phi} = \frac{Mgr}{I\omega}$$

where $r$ is the distance from the pivot to the center of mass, $\phi$ is the angle of the spin axis from vertical, and $\omega$ is the spin rate.

> **Surprising result:** A faster spin ($\omega \uparrow$) leads to **slower** precession ($\Omega_{\text{prec}} \downarrow$). The gyroscope resists change more when spinning fast!

---
## 🎮 Interactive Demo 2: Gyroscope Precession Visualizer

Adjust the spin rate and tilt angle to see how the precession changes. The 3D view shows the angular momentum vector tracing out a cone.

In [ ]:
# ============================================================
# DEMO 2: Gyroscope Precession Visualizer
# ============================================================

def gyroscope_precession(spin_rpm=3000, tilt_deg=30, duration=10.0):
    """
    Visualize gyroscopic precession.
    Shows a top view and a side view of the gyroscope axis sweeping a cone.
    """
    # Gyroscope parameters
    M = 0.5        # kg
    R_gyro = 0.05  # m (disk radius)
    r_cm = 0.08    # m (distance from pivot to CM)
    g = 9.81

    I = 0.5 * M * R_gyro**2
    omega_spin = spin_rpm * 2 * np.pi / 60
    L = I * omega_spin
    tilt = np.radians(tilt_deg)

    # Torque and precession rate
    tau = M * g * r_cm * np.sin(tilt)
    if omega_spin > 0.1 and tilt_deg > 0:
        Omega_prec = tau / (I * omega_spin)
    else:
        Omega_prec = 0

    fps = 30
    n_frames = int(duration * fps)
    times = np.linspace(0, duration, n_frames)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5),
                              gridspec_kw={'width_ratios': [1, 1, 1.2]})
    fig.suptitle(f'Gyroscope Precession — Spin = {spin_rpm} RPM, Tilt = {tilt_deg}$^\\circ$',
                 fontsize=14, fontweight='bold')

    # --- Top view (xy plane) ---
    ax_top = axes[0]
    ax_top.set_xlim(-0.15, 0.15)
    ax_top.set_ylim(-0.15, 0.15)
    ax_top.set_aspect('equal')
    ax_top.set_title('Top View (looking down)')
    ax_top.set_xlabel('x (m)')
    ax_top.set_ylabel('y (m)')
    ax_top.plot([0], [0], 'k+', markersize=15, markeredgewidth=2)  # pivot

    # Precession circle
    r_proj = r_cm * np.sin(tilt)  # projection of gyro arm on horizontal
    circle_angles = np.linspace(0, 2*np.pi, 100)
    ax_top.plot(r_proj * np.cos(circle_angles), r_proj * np.sin(circle_angles),
               'g--', alpha=0.3, lw=1)

    gyro_arm_top, = ax_top.plot([], [], 'b-', lw=3)
    gyro_disk_top, = ax_top.plot([], [], 'ro', markersize=12)
    trail_x, trail_y = [], []
    trail_top, = ax_top.plot([], [], 'r-', alpha=0.3, lw=1)

    # --- Side view (xz plane) ---
    ax_side = axes[1]
    ax_side.set_xlim(-0.15, 0.15)
    ax_side.set_ylim(-0.02, 0.15)
    ax_side.set_aspect('equal')
    ax_side.set_title('Side View')
    ax_side.set_xlabel('x (m)')
    ax_side.set_ylabel('z (m)')
    ax_side.axhline(y=0, color='brown', lw=2)
    ax_side.plot([0], [0], 'k^', markersize=10)  # pivot

    gyro_arm_side, = ax_side.plot([], [], 'b-', lw=3)
    gyro_disk_side, = ax_side.plot([], [], 'ro', markersize=12)

    # --- Info panel ---
    ax_info = axes[2]
    ax_info.axis('off')

    prec_period = 2*np.pi/Omega_prec if Omega_prec > 0 else float('inf')
    info = [
        f'Gyroscope Parameters:',
        f'  Mass M = {M:.2f} kg',
        f'  Disk radius R = {R_gyro*100:.1f} cm',
        f'  CM distance r = {r_cm*100:.1f} cm',
        f'  I = (1/2)MR$^2$ = {I:.6f} kg m$^2$',
        f'',
        f'Spin:',
        f'  $\\omega_{{spin}}$ = {spin_rpm} RPM = {omega_spin:.1f} rad/s',
        f'  L = I$\\omega$ = {L:.4f} kg m$^2$/s',
        f'',
        f'Torque:',
        f'  $\\tau$ = Mgr sin$\\phi$ = {tau:.4f} N m',
        f'',
        f'Precession:',
        f'  $\\Omega_{{prec}}$ = $\\tau$ / (I$\\omega$)',
        f'         = {Omega_prec:.3f} rad/s',
        f'         = {Omega_prec*60/(2*np.pi):.2f} RPM',
        f'  Period = {prec_period:.2f} s',
    ]
    ax_info.text(0.05, 0.95, '\n'.join(info), transform=ax_info.transAxes,
                 fontsize=10, verticalalignment='top', fontfamily='monospace',
                 bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

    time_text = ax_top.text(0.02, 0.02, '', transform=ax_top.transAxes, fontsize=9)

    plt.tight_layout()

    def update(frame):
        t = times[frame]
        phi_prec = Omega_prec * t  # precession angle

        # Gyroscope axis endpoint
        x_end = r_cm * np.sin(tilt) * np.cos(phi_prec)
        y_end = r_cm * np.sin(tilt) * np.sin(phi_prec)
        z_end = r_cm * np.cos(tilt)

        # Top view
        gyro_arm_top.set_data([0, x_end], [0, y_end])
        gyro_disk_top.set_data([x_end], [y_end])
        trail_x.append(x_end)
        trail_y.append(y_end)
        trail_top.set_data(trail_x, trail_y)

        # Side view (project onto xz plane, but rotate so we always see the current plane)
        x_side = r_cm * np.sin(tilt) * np.cos(phi_prec)
        z_side = z_end
        gyro_arm_side.set_data([0, x_side], [0, z_side])
        gyro_disk_side.set_data([x_side], [z_side])

        time_text.set_text(f't = {t:.1f} s')

        return []

    anim = FuncAnimation(fig, update, frames=n_frames, interval=1000/fps, blit=True)
    plt.close(fig)
    return HTML(anim.to_jshtml())

gyroscope_precession(spin_rpm=3000, tilt_deg=30, duration=10.0)

**Explore:** Try increasing `spin_rpm` to 6000 and see how the precession slows down. Try `tilt_deg = 60` for a more dramatic cone.

---
### ⏱️ Checkpoint 1 of 3 — Think · Pair · Explain

For **Week 10: Rotational Dynamics (II) — Angular Momentum — PHASE 4: Rotation — Same Laws, New Geometry**, name the governing principle and define its symbols with SI units.

First write a private prediction. Then explain it to a partner, revise it, and enter your final explanation below. Ask a question now if any step is unclear.


In [ ]:
checkpoint_1_response = ""  # write at least 8 words
studio_pulse(1, checkpoint_1_response)

---
## 3. Torque and Angular Acceleration

### 3.1 Torque

Torque is the rotational equivalent of force:

$$\vec{\tau} = \vec{r} \times \vec{F}$$

Magnitude: $\tau = rF\sin\theta = F \cdot d$ where $d = r\sin\theta$ is the **lever arm** (perpendicular distance from the axis to the line of action of the force).

### 3.2 Newton's Second Law for Rotation

$$\sum \tau = I\alpha$$

This relates the net torque to the angular acceleration, just like $\sum F = ma$ for linear motion.

### 3.3 Work and Power in Rotation

$$W = \tau \cdot \theta, \quad P = \tau \cdot \omega$$

> **Wrench analogy:** When you push on a wrench to loosen a bolt, you want maximum torque. That means pushing perpendicular to the wrench (maximum lever arm) and as far from the bolt as possible (maximum $r$).

---
## 🎮 Interactive Demo 3: Torque and Angular Acceleration

Apply different forces at different positions on a rod pivoted at one end. See how torque, angular acceleration, and the resulting motion change.

In [ ]:
# ============================================================
# DEMO 3: Torque and Angular Acceleration Interactive
# ============================================================

def torque_interactive(F=10.0, r_frac=0.8, angle_F=90.0, M_rod=2.0, L_rod=1.0):
    """
    Interactive torque calculator and visualizer.
    A rod is pivoted at one end. A force F is applied at position r_frac*L along the rod,
    at an angle angle_F from the rod.
    """
    g = 9.81
    I = (1/3) * M_rod * L_rod**2  # rod pivoted at end
    r_force = r_frac * L_rod
    angle_rad = np.radians(angle_F)

    # Applied torque
    tau_applied = r_force * F * np.sin(angle_rad)
    lever_arm = r_force * np.sin(angle_rad)

    # Gravity torque (acts at L/2, downward)
    tau_gravity = -M_rod * g * (L_rod / 2)  # assuming rod starts horizontal

    # Net torque and angular acceleration
    tau_net = tau_applied + tau_gravity
    alpha = tau_net / I

    fig, (ax_rod, ax_info) = plt.subplots(1, 2, figsize=(14, 5),
                                           gridspec_kw={'width_ratios': [1.2, 1]})

    # --- Rod visualization ---
    ax_rod.set_xlim(-0.3, L_rod + 0.5)
    ax_rod.set_ylim(-0.8, 0.8)
    ax_rod.set_aspect('equal')
    ax_rod.set_title('Force on Pivoted Rod (Horizontal Position)', fontsize=12)
    ax_rod.set_xlabel('x (m)')

    # Pivot
    ax_rod.plot(0, 0, 'k^', markersize=15)
    ax_rod.text(0, -0.15, 'Pivot', ha='center', fontsize=9)

    # Rod
    ax_rod.plot([0, L_rod], [0, 0], 'b-', lw=8, solid_capstyle='round', alpha=0.7)

    # Force arrow
    fx = F * np.cos(angle_rad) * 0.03  # scale for display
    fy = F * np.sin(angle_rad) * 0.03
    ax_rod.annotate('', xy=(r_force + fx, fy), xytext=(r_force, 0),
                    arrowprops=dict(arrowstyle='->', color='red', lw=2.5))
    ax_rod.text(r_force + fx + 0.05, fy, f'F = {F:.0f} N\n$\\theta$ = {angle_F:.0f}$^\\circ$',
                fontsize=10, color='red')

    # Lever arm
    if abs(np.sin(angle_rad)) > 0.01:
        ax_rod.plot([r_force, r_force], [0, lever_arm*0.5], 'g--', lw=1.5, alpha=0.7)
        ax_rod.text(r_force + 0.05, lever_arm*0.25, f'd = {lever_arm:.3f} m',
                    fontsize=9, color='green')

    # CM marker
    ax_rod.plot(L_rod/2, 0, 'ko', markersize=6)
    ax_rod.annotate('', xy=(L_rod/2, -0.2), xytext=(L_rod/2, -0.05),
                    arrowprops=dict(arrowstyle='->', color='purple', lw=1.5))
    ax_rod.text(L_rod/2 + 0.05, -0.25, f'Mg = {M_rod*g:.1f} N', fontsize=9, color='purple')

    # Direction of rotation
    if abs(alpha) > 0.01:
        direction = 'CCW $\\circlearrowleft$' if alpha > 0 else 'CW $\\circlearrowright$'
        ax_rod.text(L_rod*0.7, 0.5, f'Rotation: {direction}\n$\\alpha$ = {alpha:.2f} rad/s$^2$',
                    fontsize=11, ha='center',
                    bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

    # --- Info panel ---
    ax_info.axis('off')
    info = [
        f'Rod Properties:',
        f'  M = {M_rod:.1f} kg, L = {L_rod:.1f} m',
        f'  I (about end) = (1/3)ML$^2$ = {I:.4f} kg m$^2$',
        f'',
        f'Applied Force:',
        f'  F = {F:.1f} N at r = {r_force:.2f} m from pivot',
        f'  Angle from rod = {angle_F:.0f}$^\\circ$',
        f'  Lever arm d = r sin$\\theta$ = {lever_arm:.4f} m',
        f'',
        f'Torques:',
        f'  $\\tau_{{applied}}$ = r F sin$\\theta$ = {tau_applied:.3f} N m',
        f'  $\\tau_{{gravity}}$ = -Mg(L/2) = {tau_gravity:.3f} N m',
        f'  $\\tau_{{net}}$ = {tau_net:.3f} N m',
        f'',
        f'Result:',
        f'  $\\alpha$ = $\\tau_{{net}}$ / I = {alpha:.3f} rad/s$^2$',
        f'  $\\alpha$ = {np.degrees(alpha):.1f} deg/s$^2$',
    ]
    ax_info.text(0.05, 0.95, '\n'.join(info), transform=ax_info.transAxes,
                 fontsize=10, verticalalignment='top', fontfamily='monospace',
                 bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

    plt.tight_layout()
    plt.show()

interact(torque_interactive,
         F=FloatSlider(min=0, max=50, step=1, value=10, description='Force (N)'),
         r_frac=FloatSlider(min=0.1, max=1.0, step=0.05, value=0.8, description='r / L'),
         angle_F=FloatSlider(min=0, max=180, step=5, value=90, description='Angle (deg)'),
         M_rod=FloatSlider(min=0.5, max=5, step=0.5, value=2.0, description='Rod mass (kg)'),
         L_rod=FloatSlider(min=0.5, max=2.0, step=0.1, value=1.0, description='Rod length (m)'));

**Explore:** Set the angle to 0 or 180 degrees. What happens to the torque? This is why you push a door perpendicular to it, never along it!

---
## 4. Angular Momentum Conservation in Collisions

### 4.1 Rotational Collisions

When two rotating objects interact (or a moving object strikes a pivoted one), angular momentum about the pivot is conserved if there is no external torque.

**Example: Bullet embedding in a rotating rod**

A bullet of mass $m$ moving at speed $v$ strikes and embeds in the end of a stationary rod (mass $M$, length $L$) pivoted at the other end:

$$L_i = mvL = \left(\frac{1}{3}ML^2 + mL^2\right)\omega_f$$

$$\omega_f = \frac{mv}{(\frac{1}{3}M + m)L}$$

### 4.2 Rotational Inelastic Collision

Two disks on a common axis — one spinning, one stationary — are brought together:

$$I_1 \omega_1 = (I_1 + I_2)\omega_f$$

$$\omega_f = \frac{I_1}{I_1 + I_2}\omega_1$$

Kinetic energy is **not** conserved (friction between the disks dissipates energy), but angular momentum **is** conserved.

---
## 🎮 Interactive Demo 4: Angular Momentum Conservation in Collisions

Watch a spinning disk collide with a stationary disk. Observe how angular momentum is conserved while kinetic energy is lost.

In [ ]:
# ============================================================
# DEMO 4: Angular Momentum Conservation in Disk Collision
# ============================================================

def angular_collision_animation(M1=2.0, R1=0.3, omega1_init=10.0, M2=3.0, R2=0.25):
    """
    Animate two coaxial disks: disk 1 spinning, disk 2 stationary.
    At t_collision, they couple and spin together.
    Shows angular velocities, angular momentum, and kinetic energy.
    """
    I1 = 0.5 * M1 * R1**2
    I2 = 0.5 * M2 * R2**2
    L_total = I1 * omega1_init

    omega_final = L_total / (I1 + I2)

    KE_before = 0.5 * I1 * omega1_init**2
    KE_after = 0.5 * (I1 + I2) * omega_final**2
    KE_lost = KE_before - KE_after

    duration = 8.0
    t_collision = 3.0
    t_transition = 0.5  # coupling duration
    fps = 30
    n_frames = int(duration * fps)
    times = np.linspace(0, duration, n_frames)

    # Compute omega schedules
    omega1_schedule = np.zeros(n_frames)
    omega2_schedule = np.zeros(n_frames)
    for i, t in enumerate(times):
        if t < t_collision:
            omega1_schedule[i] = omega1_init
            omega2_schedule[i] = 0.0
        elif t < t_collision + t_transition:
            frac = (t - t_collision) / t_transition
            omega1_schedule[i] = omega1_init + frac * (omega_final - omega1_init)
            omega2_schedule[i] = frac * omega_final
        else:
            omega1_schedule[i] = omega_final
            omega2_schedule[i] = omega_final

    # Integrate angles
    dt = duration / n_frames
    theta1 = np.cumsum(omega1_schedule) * dt
    theta2 = np.cumsum(omega2_schedule) * dt

    # L and KE schedules
    L_schedule = I1 * omega1_schedule + I2 * omega2_schedule
    KE_schedule = 0.5 * I1 * omega1_schedule**2 + 0.5 * I2 * omega2_schedule**2

    fig = plt.figure(figsize=(15, 8))
    fig.suptitle('Rotational Inelastic Collision — Two Coaxial Disks', fontsize=14, fontweight='bold')

    # Layout: 2 rows, 3 columns
    ax_disk1 = fig.add_subplot(231)
    ax_disk2 = fig.add_subplot(232)
    ax_omega = fig.add_subplot(233)
    ax_L = fig.add_subplot(234)
    ax_KE = fig.add_subplot(235)
    ax_info = fig.add_subplot(236)

    # --- Disk 1 ---
    ax_disk1.set_xlim(-0.5, 0.5)
    ax_disk1.set_ylim(-0.5, 0.5)
    ax_disk1.set_aspect('equal')
    ax_disk1.set_title(f'Disk 1 (M={M1} kg, R={R1} m)', fontsize=10)
    ax_disk1.set_xticks([])
    ax_disk1.set_yticks([])
    c1 = plt.Circle((0, 0), R1, fill=True, alpha=0.3, color='steelblue', lw=2, edgecolor='steelblue')
    ax_disk1.add_patch(c1)
    spoke1, = ax_disk1.plot([], [], '-', color='steelblue', lw=3)
    omega1_text = ax_disk1.text(0, -0.42, '', ha='center', fontsize=10)

    # --- Disk 2 ---
    ax_disk2.set_xlim(-0.5, 0.5)
    ax_disk2.set_ylim(-0.5, 0.5)
    ax_disk2.set_aspect('equal')
    ax_disk2.set_title(f'Disk 2 (M={M2} kg, R={R2} m)', fontsize=10)
    ax_disk2.set_xticks([])
    ax_disk2.set_yticks([])
    c2 = plt.Circle((0, 0), R2, fill=True, alpha=0.3, color='coral', lw=2, edgecolor='coral')
    ax_disk2.add_patch(c2)
    spoke2, = ax_disk2.plot([], [], '-', color='coral', lw=3)
    omega2_text = ax_disk2.text(0, -0.42, '', ha='center', fontsize=10)

    # --- Omega plot ---
    ax_omega.set_xlim(0, duration)
    ax_omega.set_ylim(0, omega1_init * 1.15)
    ax_omega.set_xlabel('Time (s)')
    ax_omega.set_ylabel(r'$\omega$ (rad/s)')
    ax_omega.set_title('Angular Velocities')
    line_w1, = ax_omega.plot([], [], 'b-', lw=2, label=r'$\omega_1$')
    line_w2, = ax_omega.plot([], [], 'r-', lw=2, label=r'$\omega_2$')
    ax_omega.axvline(t_collision, color='gray', ls='--', alpha=0.5)
    ax_omega.axhline(omega_final, color='green', ls=':', alpha=0.5, label=f'$\\omega_f$ = {omega_final:.2f}')
    ax_omega.legend(fontsize=8)
    ax_omega.grid(True, alpha=0.3)

    # --- L plot ---
    ax_L.set_xlim(0, duration)
    ax_L.set_ylim(0, L_total * 1.3)
    ax_L.set_xlabel('Time (s)')
    ax_L.set_ylabel(r'$L$ (kg m$^2$/s)')
    ax_L.set_title('Angular Momentum (CONSERVED)')
    line_L, = ax_L.plot([], [], 'g-', lw=2.5)
    ax_L.axhline(L_total, color='green', ls='--', alpha=0.3)
    ax_L.axvline(t_collision, color='gray', ls='--', alpha=0.5)
    ax_L.grid(True, alpha=0.3)

    # --- KE plot ---
    ax_KE.set_xlim(0, duration)
    ax_KE.set_ylim(0, KE_before * 1.3)
    ax_KE.set_xlabel('Time (s)')
    ax_KE.set_ylabel('KE (J)')
    ax_KE.set_title('Kinetic Energy (NOT conserved)')
    line_KE, = ax_KE.plot([], [], 'm-', lw=2.5)
    ax_KE.axvline(t_collision, color='gray', ls='--', alpha=0.5)
    ax_KE.axhline(KE_after, color='red', ls=':', alpha=0.5, label=f'KE after = {KE_after:.2f} J')
    ax_KE.legend(fontsize=8)
    ax_KE.grid(True, alpha=0.3)

    # --- Info ---
    ax_info.axis('off')
    info = [
        f'Before collision:',
        f'  $\\omega_1$ = {omega1_init:.1f} rad/s',
        f'  $\\omega_2$ = 0 rad/s',
        f'  L = {L_total:.3f} kg m$^2$/s',
        f'  KE = {KE_before:.3f} J',
        f'',
        f'After collision:',
        f'  $\\omega_f$ = L/(I$_1$+I$_2$) = {omega_final:.3f} rad/s',
        f'  L = {L_total:.3f} kg m$^2$/s (same!)',
        f'  KE = {KE_after:.3f} J',
        f'',
        f'Energy lost = {KE_lost:.3f} J ({KE_lost/KE_before*100:.1f}%)',
        f'(dissipated as heat by friction)',
    ]
    ax_info.text(0.05, 0.95, '\n'.join(info), transform=ax_info.transAxes,
                 fontsize=10, verticalalignment='top', fontfamily='monospace',
                 bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

    plt.tight_layout()

    def update(frame):
        i = frame
        t = times[i]

        # Disk spokes
        spoke1.set_data([0, R1*0.9*np.cos(theta1[i])], [0, R1*0.9*np.sin(theta1[i])])
        spoke2.set_data([0, R2*0.9*np.cos(theta2[i])], [0, R2*0.9*np.sin(theta2[i])])

        omega1_text.set_text(f'$\\omega_1$ = {omega1_schedule[i]:.1f} rad/s')
        omega2_text.set_text(f'$\\omega_2$ = {omega2_schedule[i]:.1f} rad/s')

        # Plots
        line_w1.set_data(times[:i+1], omega1_schedule[:i+1])
        line_w2.set_data(times[:i+1], omega2_schedule[:i+1])
        line_L.set_data(times[:i+1], L_schedule[:i+1])
        line_KE.set_data(times[:i+1], KE_schedule[:i+1])

        return []

    anim = FuncAnimation(fig, update, frames=n_frames, interval=1000/fps, blit=True)
    plt.close(fig)
    return HTML(anim.to_jshtml())

angular_collision_animation(M1=2.0, R1=0.3, omega1_init=10.0, M2=3.0, R2=0.25)

**Key observations:**
- Angular momentum (green plot) is perfectly constant throughout
- Kinetic energy (purple plot) drops at the collision — energy is lost to friction/heat
- This is exactly analogous to a perfectly inelastic linear collision (like two cars sticking together)

---
## 🔧 Worked Example 1: Conservation of Angular Momentum

**Problem:** A merry-go-round (solid disk, mass 200 kg, radius 2.0 m) is spinning at 0.5 rev/s. A 50 kg child standing at the edge walks to the center. Find the new angular velocity.

In [ ]:
# Worked Example 1: Merry-go-round

M = 200.0   # kg (merry-go-round)
R = 2.0     # m
m_child = 50.0  # kg
omega_i = 0.5 * 2 * np.pi  # rad/s (0.5 rev/s)

# Initial: child at edge
I_mgr = 0.5 * M * R**2
I_child_i = m_child * R**2
I_i = I_mgr + I_child_i

# Final: child at center
I_child_f = 0  # at center, r = 0
I_f = I_mgr + I_child_f

# Conservation of angular momentum
L = I_i * omega_i
omega_f = L / I_f

print("=== Merry-Go-Round Problem ===")
print(f"\nMoments of inertia:")
print(f"  I_merry-go-round = (1/2)MR^2 = {I_mgr:.1f} kg m^2")
print(f"  I_child (at edge) = mR^2 = {I_child_i:.1f} kg m^2")
print(f"  I_initial (total) = {I_i:.1f} kg m^2")
print(f"  I_final (child at center) = {I_f:.1f} kg m^2")

print(f"\nAngular momentum:")
print(f"  L = I_i * omega_i = {I_i:.1f} * {omega_i:.2f} = {L:.1f} kg m^2/s")

print(f"\nFinal angular velocity:")
print(f"  omega_f = L / I_f = {L:.1f} / {I_f:.1f} = {omega_f:.3f} rad/s")
print(f"         = {omega_f/(2*np.pi):.3f} rev/s")
print(f"\nSpeed-up factor: omega_f / omega_i = {omega_f/omega_i:.2f}")

# Energy comparison
KE_i = 0.5 * I_i * omega_i**2
KE_f = 0.5 * I_f * omega_f**2
print(f"\nKinetic energy:")
print(f"  KE_initial = {KE_i:.1f} J")
print(f"  KE_final = {KE_f:.1f} J")
print(f"  Energy INCREASED by {KE_f - KE_i:.1f} J")
print(f"  (This energy came from the child doing work by walking inward)")

---
### ⏱️ Checkpoint 2 of 3 — Think · Pair · Explain

Before calculating, predict the direction, sign, or trend of the result. Cite the physical law and one limiting case that supports your prediction.

First write a private prediction. Then explain it to a partner, revise it, and enter your final explanation below. Ask a question now if any step is unclear.


In [ ]:
checkpoint_2_response = ""  # write at least 8 words
studio_pulse(2, checkpoint_2_response)

---
## 🔧 Worked Example 2: Torque and Angular Acceleration

**Problem:** A 3 kg solid disk of radius 0.2 m is free to rotate about its central axis. A constant tangential force of 5 N is applied at the rim. Starting from rest:
(a) What is the angular acceleration?
(b) What is the angular velocity after 4 seconds?
(c) How much work is done by the force in 4 seconds?

In [ ]:
# Worked Example 2: Torque on a disk

m = 3.0     # kg
R = 0.2     # m
F = 5.0     # N
t = 4.0     # s

I = 0.5 * m * R**2
tau = F * R  # tangential force at rim

# (a) Angular acceleration
alpha = tau / I
print(f"(a) I = (1/2)mR^2 = {I:.4f} kg m^2")
print(f"    tau = F*R = {F}*{R} = {tau:.2f} N m")
print(f"    alpha = tau/I = {tau}/{I} = {alpha:.2f} rad/s^2")

# (b) Angular velocity after 4 s
omega = alpha * t
print(f"\n(b) omega = alpha * t = {alpha:.2f} * {t} = {omega:.2f} rad/s")
print(f"    That's {omega/(2*np.pi):.2f} rev/s = {omega/(2*np.pi)*60:.0f} RPM")

# (c) Work done
theta = 0.5 * alpha * t**2
W = tau * theta
KE = 0.5 * I * omega**2
print(f"\n(c) theta = (1/2)*alpha*t^2 = {theta:.2f} rad = {theta/(2*np.pi):.1f} revolutions")
print(f"    W = tau * theta = {tau:.2f} * {theta:.2f} = {W:.2f} J")
print(f"    Cross-check: KE = (1/2)I*omega^2 = {KE:.2f} J (matches W by work-energy theorem)")

---
## 5. Unit and Scale Verification

In rotational dynamics, keeping track of units is critical. Here is a quick reference for checking your work:

| Quantity | Unit | Typical Magnitudes |
|:---|:---|:---|
| $\omega$ | rad/s | Ceiling fan: ~10; car engine: ~600; CD player: ~50 |
| $\alpha$ | rad/s$^2$ | Typically 0.1 to 100 for everyday objects |
| $I$ | kg m$^2$ | Baseball bat: ~0.5; bicycle wheel: ~0.1; human body: ~1-10 |
| $\tau$ | N m | Opening a door: ~1-5; car engine: ~200-500 |
| $L$ | kg m$^2$/s | Same units as $I \times \omega$ |

### Quick Conversion
- RPM to rad/s: multiply by $2\pi/60$
- rev/s to rad/s: multiply by $2\pi$
- degrees to radians: multiply by $\pi/180$

---
## Problem Set

**Instructions:** Solve each problem analytically first, then verify your answer numerically in the code cell below it. Show your work with clear variable definitions and unit tracking.

- **L1 (Basic):** Straightforward single-concept problems
- **L2 (Intermediate):** Multi-step problems combining two concepts
- **L3 (Challenge):** Multi-concept integration and engineering applications

---
## 🧪 Practice Priority

**L1 problems are the core in-class set.** L2 problems are extensions if time remains; L3 problems are challenges. Nothing here is homework or collected. For every solution use: diagram/configuration → governing law → symbolic setup → units → numerical result → reasonableness check.


### L1 (Basic) — P1

A solid disk of mass $3.0$ kg and radius $0.25$ m rotates at $10$ rad/s about its central axis. Calculate its angular momentum.

<details><summary>Answer</summary>$L = 0.938$ kg$\cdot$m$^2$/s</details>

In [ ]:
# ✏️ [P1] Your solution here

### L1 (Basic) — P2

A $0.15$ kg ball moves in a circle of radius $0.80$ m at a speed of $4.0$ m/s. Find the magnitude of its angular momentum about the center of the circle.

<details><summary>Answer</summary>$L = 0.48$ kg$\cdot$m$^2$/s</details>

In [ ]:
# ✏️ [P2] Your solution here

### L1 (Basic) — P3

A torque of $12$ N$\cdot$m is applied to a wheel for $3.0$ s. If the wheel starts from rest, what is its final angular momentum?

<details><summary>Answer</summary>$L = 36$ kg$\cdot$m$^2$/s</details>

In [ ]:
# ✏️ [P3] Your solution here

### L1 (Basic) — P4

A figure skater with $I = 4.6$ kg$\cdot$m$^2$ spins at $1.5$ rev/s. She pulls her arms in, reducing her moment of inertia to $1.8$ kg$\cdot$m$^2$. Find her new angular velocity in rev/s.

<details><summary>Answer</summary>$\omega_f = 3.83$ rev/s</details>

In [ ]:
# ✏️ [P4] Your solution here

---
### ⏱️ Checkpoint 3 of 3 — Think · Pair · Explain

Select one L1 solution and explain its diagram, governing law, units, and reasonableness check. Then connect the result to either the Mechatronics or Computer Engineering lens above.

First write a private prediction. Then explain it to a partner, revise it, and enter your final explanation below. Ask a question now if any step is unclear.


In [ ]:
checkpoint_3_response = ""  # write at least 8 words
studio_pulse(3, checkpoint_3_response)

---
## 🌟 Extension Problems

L2 and L3 problems are optional enrichment, not homework. Use them for remaining studio time or independent curiosity.


### L2 (Intermediate) — P5

A merry-go-round (solid disk, $M = 150$ kg, $R = 2.0$ m) is spinning freely at $0.40$ rev/s. A $40$ kg child running at $3.0$ m/s tangentially jumps onto the rim. Find (a) the angular momentum of the child about the center just before landing and (b) the new angular velocity of the system after the child lands.

<details><summary>Answer</summary>(a) $L_\text{child} = mvR = 40(3.0)(2.0) = 240$ kg$\cdot$m$^2$/s. (b) $I_\text{disk} = \tfrac12(150)(2.0)^2 = 300$ kg$\cdot$m$^2$, $\omega_i = 0.40(2\pi) = 2.5133$ rad/s, so $\omega_f = \dfrac{300(2.5133) + 240}{300 + 40(2.0)^2} = \dfrac{993.98}{460} = 2.161$ rad/s $= 0.344$ rev/s. **[CORRECTED]** (b) previously 2.72 rad/s (0.433 rev/s).</details>

In [ ]:
# ✏️ [P5] Your solution here

### L2 (Intermediate) — P6

Two coaxial disks can rotate freely. Disk A ($I_A = 0.50$ kg$\cdot$m$^2$) spins at $8.0$ rad/s. Disk B ($I_B = 0.30$ kg$\cdot$m$^2$) spins at $-5.0$ rad/s (opposite direction). They are brought together and reach a common angular velocity. Find (a) the final angular velocity and (b) the fraction of kinetic energy lost.

<details><summary>Answer</summary>(a) $\omega_f = 3.125$ rad/s; (b) $KE_i = 16.00 + 3.75 = 19.75$ J, $KE_f = \tfrac12(0.80)(3.125)^2 = 3.906$ J, so $15.844/19.75 = 80.2\%$ of the kinetic energy is lost. Cross-check: $\tfrac12\frac{I_AI_B}{I_A+I_B}(\omega_A-\omega_B)^2 = 15.844$ J. **[CORRECTED]** (b) previously 83.8%, which is inconsistent with its own $\omega_f$.</details>

In [ ]:
# ✏️ [P6] Your solution here

### L2 (Intermediate) — P7

A $0.020$ kg bullet traveling at $400$ m/s strikes and embeds in a $2.0$ kg thin rod of length $1.0$ m at its tip. The rod is pivoted at its other end and initially at rest. Find (a) the angular velocity of the rod just after impact and (b) the maximum angle the rod swings upward. Take $g = 9.81$ m/s$^2$.

<details><summary>Answer</summary>(a) $I = \tfrac13(2.0)(1.0)^2 + (0.020)(1.0)^2 = 0.68667$ kg$\cdot$m$^2$, so $\omega = \dfrac{m_b v L}{I} = \dfrac{8.0}{0.68667} = 11.65$ rad/s. (b) **There is no maximum angle.** $KE = \tfrac12 I\omega^2 = 46.60$ J, while lifting the rod and bullet all the way over the top ($\theta = 180^\circ$) needs only $(Mg\tfrac{L}{2} + m_b gL)(2) = 20.01$ J. The rod therefore rotates continuously. **[CORRECTED]** previously $\omega = 11.32$ rad/s and $\theta_\text{max} = 67.8^\circ$; that angle is unreachable, and even the old $\omega$ gives more than twice the energy needed to go over the top.</details>

In [ ]:
# ✏️ [P7] Your solution here

### L2 (Intermediate) — P8

A neutron star collapses from an initial radius of $R_i = 7.0 \times 10^5$ km (like the Sun) with rotation period $T_i = 30$ days to a final radius of $R_f = 12$ km. Assuming uniform density and conservation of angular momentum, find the final rotation period.

<details><summary>Answer</summary>At fixed mass and uniform density $I \propto R^2$, so $T \propto R^2$: $T_f = T_i(R_f/R_i)^2 = 2.592\times10^6 (12/7\times10^5)^2 = 7.617\times10^{-4}$ s $= 0.762$ ms, i.e. about 1310 rev/s. **[CORRECTED]** previously 2.5 ms (400 rev/s), which corresponds to $R_f \approx 21.7$ km.</details>

In [ ]:
# ✏️ [P8] Your solution here

### L3 (Challenge) — P9

A turntable ($I = 0.015$ kg$\cdot$m$^2$) rotates freely at $33\frac{1}{3}$ RPM. A $0.020$ kg ring of putty is dropped vertically onto the turntable at a radius of $0.12$ m. The putty sticks. (a) Find the new angular velocity. (b) A motor then applies a constant torque to return the turntable to its original speed in $2.0$ s. Find the required torque and the work done by the motor.

<details><summary>Answer</summary>(a) $\omega_f = 3.30$ rad/s; (b) $\tau = 0.00544$ N$\cdot$m; $W = 0.0499$ J</details>

In [ ]:
# ✏️ [P9] Your solution here

### L3 (Challenge) — P10

A spacecraft uses a single reaction wheel ($I_w = 0.80$ kg$\cdot$m$^2$) for attitude control about one axis. The spacecraft body has $I_s = 120$ kg$\cdot$m$^2$. The wheel is initially at rest. (a) To rotate the spacecraft by $15^\circ$ in $60$ s at constant angular velocity, what constant angular velocity must the wheel maintain? (b) If the wheel has a maximum speed of $6000$ RPM, what is the maximum angular velocity of the spacecraft? (c) How much energy does the motor consume for part (a)?

<details><summary>Answer</summary>(a) $\omega_w = 0.655$ rad/s ($6.25$ RPM); (b) $\omega_{s,\text{max}} = 4.19$ rad/s; (c) $E = 0.172$ J</details>

In [ ]:
# ✏️ [P10] Your solution here

---
## 🌉 Bridge to Next Week

This week we explored angular momentum, its conservation, and its applications to collisions and gyroscopes. These principles appear everywhere in engineering:

- **Spacecraft attitude control** uses reaction wheels (conservation of angular momentum)
- **Gyroscopes** stabilize ships, aircraft, and smartphones
- **Flywheels** store energy as rotational kinetic energy

Next week, we will move into **oscillatory motion** — systems that go back and forth repeatedly. Springs, pendulums, and even molecular vibrations all share the same mathematical description: **simple harmonic motion**. You will see how energy continuously converts between kinetic and potential forms, creating the rhythmic patterns that are fundamental to waves and sound.